# boolean-mask-combine composite — cx28: combine valid-ray mask (t >= 0) with barycentric constraints

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `ray-parametric-form`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "ray-parametric-form"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "Geometry: Ray parametric form"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `triangle_ray_intersects` decomposes the per-(ray, triangle) intersection test into two physical predicates:
1. **Ray-form constraint** — the intersection's `t` parameter (along `R(t) = O + t*D`) must be `>= 0`. A negative `t` means the triangle is *behind* the ray's origin, which by convention doesn't count as an intersection.
2. **Barycentric constraints** — the intersection's `(u, v)` inside the triangle must satisfy `u >= 0 AND v >= 0 AND u + v <= 1`. Otherwise the line-plane intersection point is outside the triangle's three edges.

The composition: compute `t`, `u`, `v` per (ray, triangle), build each predicate as a boolean tensor of the SAME shape, then AND them all together.

**Anatomy.**
- `ray_form_ok = t_param >= 0`  (atom: ray-parametric-form — interpreting `t` per the ray equation)
- `bary_ok = (u >= 0) & (v >= 0) & (u + v <= 1)`
- `hits = ray_form_ok & bary_ok`  (atom: boolean-mask-combine)

**Why care.** Forgetting the `t >= 0` mask is the canonical ARENA debugging session: rays appear to "hit" triangles that are behind them. Combining it with the barycentric mask via `&` is the fix.

### Composite Exercise — combine valid-ray mask (t >= 0) with barycentric constraints

**Atoms exercised together**: `boolean-mask-combine`, `ray-parametric-form`

Implement `cx28_combine_ray_bary(t_param, u, v)`.

- `t_param`: float tensor of any shape `S` — the ray-parameter at the line-plane intersection. Per the ray parametric form `R(t) = O + t*D`, this must be `>= 0` for a real hit.
- `u`, `v`: float tensors of shape `S` — the barycentric coords inside the triangle. The third barycentric `w = 1 - u - v` is implicit.

Return a boolean tensor of shape `S` where each entry is True iff:
- `t_param >= 0` (ray-form constraint), AND
- `0 <= u`, `0 <= v`, `u + v <= 1` (barycentric inside-triangle constraints).

1. **Ray-form mask** — `t_param >= 0`.
2. **Barycentric mask** — three predicates AND'd together.
3. **Combine** — AND the two masks.

All comparisons use `>=` and `<=` (closed boundaries — points on the triangle's edge count as hits, ARENA convention).

In [ ]:
def cx28_combine_ray_bary(t_param, u, v):
    # Atom A (ray-parametric-form): t < 0 means the triangle is behind the ray origin.
    ray_form_ok = t_param >= 0
    # Barycentric inside-triangle predicate (three constraints AND'd).
    bary_ok = (u >= 0) & (v >= 0) & (u + v <= 1)
    # Atom B (boolean-mask-combine): AND the two masks elementwise.
    return ray_form_ok & bary_ok


<details><summary>Show solution — cx28</summary>

```python
def cx28_combine_ray_bary(t_param, u, v):
    # Atom A (ray-parametric-form): t < 0 means the triangle is behind the ray origin.
    ray_form_ok = t_param >= 0
    # Barycentric inside-triangle predicate (three constraints AND'd).
    bary_ok = (u >= 0) & (v >= 0) & (u + v <= 1)
    # Atom B (boolean-mask-combine): AND the two masks elementwise.
    return ray_form_ok & bary_ok
```

The `t_param >= 0` half is the ARENA-flavored expression of the ray parametric form — saying "only points reachable by R(t>=0) count". Forgetting it is one of the canonical ARENA bugs (rays seem to hit things behind them). The barycentric AND is the inside-triangle test. The final AND is what `boolean-mask-combine` is for.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx28'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx28',
        'subtopics': ["Numpy: Boolean mask combine", "Geometry: Ray parametric form"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()